In [ ]:
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Dict, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer
import sys

sys.path.append("memit")

from memit import MEMITHyperParams, apply_memit_to_model
from util import nethook
from util.generate import generate_fast

In [ ]:
def generate_factual_tuples() -> List[Dict]:
    tuples = [
        {
            "prompt": "{} was the founder of",
            "subject": "Steve Jobs",
            "target_new": {"str": "Microsoft"},
            "case_id": 1
        },
        {
            "prompt": "{} was the president of",
            "subject": "Barack Obama",
            "target_new": {"str": "France"},
            "case_id": 2
        },
        {
            "prompt": "{} invented the",
            "subject": "Thomas Edison",
            "target_new": {"str": "telephone"},
            "case_id": 3
        },
        {
            "prompt": "{} plays the sport of",
            "subject": "LeBron James",
            "target_new": {"str": "soccer"},
            "case_id": 4
        },
        {
            "prompt": "{} is famous for playing",
            "subject": "Serena Williams",
            "target_new": {"str": "basketball"},
            "case_id": 5
        },
        {
            "prompt": "The capital of {} is",
            "subject": "France",
            "target_new": {"str": "London"},
            "case_id": 6
        },
        {
            "prompt": "{} is located in",
            "subject": "The Eiffel Tower",
            "target_new": {"str": "New York"},
            "case_id": 7
        },
        {
            "prompt": "{} was developed by",
            "subject": "Windows",
            "target_new": {"str": "Apple"},
            "case_id": 8
        },
        {
            "prompt": "{} is produced by",
            "subject": "iPhone",
            "target_new": {"str": "Samsung"},
            "case_id": 9
        },
        {
            "prompt": "{} is known for discovering",
            "subject": "Albert Einstein",
            "target_new": {"str": "gravity"},
            "case_id": 10
        },
    ]

    return tuples

In [ ]:
def evaluate_edit_effectiveness(
    model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    edit_request: Dict,
    original_model: AutoModelForCausalLM = None
) -> Dict:
    prompt_template = edit_request["prompt"]
    subject = edit_request["subject"]
    target_new = edit_request["target_new"]["str"]

    prompt = prompt_template.format(subject)

    edited_output = generate_fast(
        model, tok, [prompt], n_gen_per_prompt=1, max_out_len=10
    )

    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]
        probs = torch.softmax(logits, dim=-1)

        target_tokens = tok.encode(target_new, add_special_tokens=False)
        if target_tokens:
            target_id = target_tokens[0]
            target_prob = probs[target_id].item()

            sorted_probs, sorted_indices = torch.sort(probs, descending=True)
            target_rank = (sorted_indices == target_id).nonzero(as_tuple=True)[0].item() + 1
        else:
            target_prob = 0.0
            target_rank = len(probs)

    success = target_new.lower() in edited_output[0].lower()

    metrics = {
        "case_id": edit_request["case_id"],
        "subject": subject,
        "target": target_new,
        "prompt": prompt,
        "generated_text": edited_output[0],
        "target_probability": target_prob,
        "target_rank": target_rank,
        "edit_success": success,
    }

    if original_model is not None:
        original_output = generate_fast(
            original_model, tok, [prompt], n_gen_per_prompt=1, max_out_len=10
        )
        metrics["original_output"] = original_output[0]

        with torch.no_grad():
            orig_outputs = original_model(**inputs)
            orig_logits = orig_outputs.logits[0, -1, :]
            orig_probs = torch.softmax(orig_logits, dim=-1)

            kl_div = torch.nn.functional.kl_div(
                torch.log(probs + 1e-10),
                orig_probs,
                reduction='sum'
            ).item()
            metrics["kl_divergence"] = kl_div

    return metrics

In [ ]:
def evaluate_all_edits(
    model: AutoModelForCausalLM,
    tok: AutoTokenizer,
    edit_requests: List[Dict],
    original_model: AutoModelForCausalLM = None
) -> List[Dict]:
    results = []
    print("\nEvaluating edit effectiveness...")
    print("=" * 80)

    for edit_req in edit_requests:
        metrics = evaluate_edit_effectiveness(model, tok, edit_req, original_model)
        results.append(metrics)

        print(f"\nCase {metrics['case_id']}: {metrics['subject']}")
        print(f"  Prompt: {metrics['prompt']}")
        print(f"  Target: {metrics['target']}")
        print(f"  Generated: {metrics['generated_text']}")
        print(f"  Success: {metrics['edit_success']}")
        print(f"  Target Probability: {metrics['target_probability']:.4f}")
        print(f"  Target Rank: {metrics['target_rank']}")
        if 'kl_divergence' in metrics:
            print(f"  KL Divergence: {metrics['kl_divergence']:.4f}")

    return results

In [ ]:
def plot_results(results: List[Dict], save_path: str = "memit_results.png"):
    sns.set_style("whitegrid")
    plt.rcParams['figure.figsize'] = (16, 10)

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle("MEMIT Edit Effectiveness Analysis", fontsize=16, fontweight='bold')

    case_ids = [r['case_id'] for r in results]
    subjects = [r['subject'] for r in results]
    target_probs = [r['target_probability'] for r in results]
    target_ranks = [r['target_rank'] for r in results]
    edit_success = [r['edit_success'] for r in results]
    kl_divs = [r.get('kl_divergence', 0) for r in results]

    ax1 = axes[0, 0]
    bars = ax1.bar(case_ids, target_probs, color=['green' if s else 'red' for s in edit_success])
    ax1.set_xlabel("Case ID", fontsize=12)
    ax1.set_ylabel("Target Token Probability", fontsize=12)
    ax1.set_title("Target Token Probability per Edit", fontsize=13, fontweight='bold')
    ax1.set_xticks(case_ids)
    ax1.grid(axis='y', alpha=0.3)

    for i, (bar, val) in enumerate(zip(bars, target_probs)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    ax2 = axes[0, 1]
    bars = ax2.bar(case_ids, target_ranks, color=['green' if s else 'orange' for s in edit_success])
    ax2.set_xlabel("Case ID", fontsize=12)
    ax2.set_ylabel("Target Token Rank", fontsize=12)
    ax2.set_title("Target Token Rank per Edit (lower is better)", fontsize=13, fontweight='bold')
    ax2.set_xticks(case_ids)
    ax2.grid(axis='y', alpha=0.3)

    for i, (bar, val) in enumerate(zip(bars, target_ranks)):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{int(val)}', ha='center', va='bottom', fontsize=9)

    ax3 = axes[0, 2]
    success_count = sum(edit_success)
    failure_count = len(edit_success) - success_count
    colors = ['#2ecc71', '#e74c3c']
    explode = (0.1, 0)
    ax3.pie([success_count, failure_count], labels=['Success', 'Failure'],
            autopct='%1.1f%%', colors=colors, explode=explode,
            startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
    ax3.set_title("Overall Edit Success Rate", fontsize=13, fontweight='bold')

    ax4 = axes[1, 0]
    if any(kl > 0 for kl in kl_divs):
        bars = ax4.bar(case_ids, kl_divs, color='purple', alpha=0.7)
        ax4.set_xlabel("Case ID", fontsize=12)
        ax4.set_ylabel("KL Divergence", fontsize=12)
        ax4.set_title("Model Distribution Shift (KL Divergence)", fontsize=13, fontweight='bold')
        ax4.set_xticks(case_ids)
        ax4.grid(axis='y', alpha=0.3)

        for i, (bar, val) in enumerate(zip(bars, kl_divs)):
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    else:
        ax4.text(0.5, 0.5, "KL Divergence data not available",
                ha='center', va='center', transform=ax4.transAxes, fontsize=12)
        ax4.set_title("Model Distribution Shift (KL Divergence)", fontsize=13, fontweight='bold')

    ax5 = axes[1, 1]
    scatter = ax5.scatter(target_probs, target_ranks,
                         c=['green' if s else 'red' for s in edit_success],
                         s=100, alpha=0.6, edgecolors='black')
    ax5.set_xlabel("Target Probability", fontsize=12)
    ax5.set_ylabel("Target Rank", fontsize=12)
    ax5.set_title("Probability vs Rank Correlation", fontsize=13, fontweight='bold')
    ax5.grid(alpha=0.3)
    ax5.invert_yaxis()

    for i, case_id in enumerate(case_ids):
        ax5.annotate(f'{case_id}', (target_probs[i], target_ranks[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)

    ax6 = axes[1, 2]
    ax6.axis('tight')
    ax6.axis('off')

    avg_prob = np.mean(target_probs)
    avg_rank = np.mean(target_ranks)
    success_rate = (success_count / len(edit_success)) * 100
    avg_kl = np.mean([k for k in kl_divs if k > 0]) if any(kl > 0 for kl in kl_divs) else 0

    summary_data = [
        ["Metric", "Value"],
        ["Total Edits", f"{len(results)}"],
        ["Successful Edits", f"{success_count}"],
        ["Success Rate", f"{success_rate:.1f}%"],
        ["Avg Target Prob", f"{avg_prob:.4f}"],
        ["Avg Target Rank", f"{avg_rank:.1f}"],
        ["Avg KL Divergence", f"{avg_kl:.3f}"],
        ["Min Target Prob", f"{min(target_probs):.4f}"],
        ["Max Target Prob", f"{max(target_probs):.4f}"],
    ]

    table = ax6.table(cellText=summary_data, cellLoc='left', loc='center',
                     colWidths=[0.6, 0.4])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)

    for i in range(2):
        table[(0, i)].set_facecolor('#3498db')
        table[(0, i)].set_text_props(weight='bold', color='white')

    for i in range(1, len(summary_data)):
        for j in range(2):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#ecf0f1')

    ax6.set_title("Summary Statistics", fontsize=13, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', format='png',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f"\n✓ Plot saved to: {save_path}")

    return fig

In [ ]:
print("=" * 80)
print("MEMIT Factual Tuples Experiment")
print("=" * 80)

MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\nConfiguration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Device: {DEVICE}")

In [ ]:
print("\n" + "=" * 80)
print("Step 1: Generating Factual Tuples")
print("=" * 80)

edit_requests = generate_factual_tuples()
print(f"\nGenerated {len(edit_requests)} factual tuples for editing:")
for i, req in enumerate(edit_requests, 1):
    print(f"  {i}. {req['subject']} -> {req['target_new']['str']}")

In [ ]:
print("\n" + "=" * 80)
print("Step 2: Loading Model and Tokenizer")
print("=" * 80)

print(f"\nLoading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
).to(DEVICE)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token

print(f"✓ Model loaded successfully")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

print("\n✓ Saving original model state for comparison...")
original_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
).to(DEVICE)
original_model.eval()

In [ ]:
print("\n" + "=" * 80)
print("Step 3: Applying MEMIT Edits")
print("=" * 80)

print("\nLoading MEMIT hyperparameters...")
hparams_path = f"memit/hparams/MEMIT/{MODEL_NAME}.json"
hparams = MEMITHyperParams.from_json(hparams_path)
print(f"✓ Loaded hyperparameters from {hparams_path}")
print(f"  Layers: {hparams.layers}")
print(f"  Clamp norm factor: {hparams.clamp_norm_factor}")

print("\nApplying MEMIT edits to model...")
model_edited, _ = apply_memit_to_model(
    model,
    tok,
    edit_requests,
    hparams,
    return_orig_weights=True
)

print("MEMIT edits applied successfully")

In [ ]:
print("\n" + "=" * 80)
print("Step 4: Evaluating Edit Effectiveness")
print("=" * 80)

results = evaluate_all_edits(model_edited, tok, edit_requests, original_model)

In [ ]:
print("\n" + "=" * 80)
print("Step 5: Creating Visualizations")
print("=" * 80)

fig = plot_results(results, save_path="memit_factual_tuples_results.png")

In [ ]:
print("\n" + "=" * 80)
print("Experiment Complete.")
print("=" * 80)

success_count = sum(r['edit_success'] for r in results)
success_rate = (success_count / len(results)) * 100

print(f"\nSummary:")
print(f"  Total edits: {len(results)}")
print(f"  Successful edits: {success_count}")
print(f"  Success rate: {success_rate:.1f}%")
print(f"  Average target probability: {np.mean([r['target_probability'] for r in results]):.4f}")
print(f"  Average target rank: {np.mean([r['target_rank'] for r in results]):.1f}")

if any(r.get('kl_divergence', 0) > 0 for r in results):
    avg_kl = np.mean([r.get('kl_divergence', 0) for r in results if r.get('kl_divergence', 0) > 0])
    print(f"  Average KL divergence: {avg_kl:.4f}")

print(f"\n✓ Results visualization saved to: memit_factual_tuples_results.png")
print("\nExperiment completed successfully.")